In [1]:
# Importing libraries

import xarray as xr
import glob
import numpy as np
import dask.dataframe as dd
from dask.distributed import Client, LocalCluster
import dask.array as da
import lightgbm as lgb
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import matplotlib.pyplot as plt
import seaborn as sns
import os
from lightgbm import DaskLGBMRegressor

In [2]:
path = "C:/Users/idasi/Documents/ideas_tih/IDEAS-TIH-Summer-2026/data/processed"
files = sorted((glob.glob(os.path.join(path, "*.nc"))))
print(f"Total files found: {len(files)}")

Total files found: 120


In [3]:
dset = xr.open_mfdataset(
    files,
    parallel=True,
    engine='netcdf4',
    data_vars='minimal',
    chunks={'valid_time': 720, 'latitude': 125, 'longitude': 121}
    )

print(dset)

print("Chunks:", dset['t2m'].chunks)
print("Type:", type(dset['t2m'].data))

c:\Users\idasi\.conda\envs\ideastih\Lib\site-packages\dask\_task_spec.py:767: UserWarning: The specified chunks separate the stored chunks along dimension "valid_time" starting at index 720. This could degrade performance. Instead, consider rechunking after loading.
  return self.func(*new_argspec, **kwargs)
c:\Users\idasi\.conda\envs\ideastih\Lib\site-packages\dask\_task_spec.py:767: UserWarning: The specified chunks separate the stored chunks along dimension "valid_time" starting at index 720. This could degrade performance. Instead, consider rechunking after loading.
  return self.func(*new_argspec, **kwargs)
c:\Users\idasi\.conda\envs\ideastih\Lib\site-packages\dask\_task_spec.py:767: UserWarning: The specified chunks separate the stored chunks along dimension "valid_time" starting at index 720. This could degrade performance. Instead, consider rechunking after loading.
  return self.func(*new_argspec, **kwargs)
c:\Users\idasi\.conda\envs\ideastih\Lib\site-packages\dask\_task_spec.

<xarray.Dataset> Size: 37GB
Dimensions:     (valid_time: 87672, latitude: 125, longitude: 121)
Coordinates:
  * valid_time  (valid_time) datetime64[ns] 701kB 2016-01-01 ... 2025-12-31T2...
    expver      (valid_time) <U4 1MB dask.array<chunksize=(720,), meta=np.ndarray>
  * latitude    (latitude) float64 1kB 36.0 35.75 35.5 35.25 ... 5.5 5.25 5.0
  * longitude   (longitude) float64 968B 63.0 63.25 63.5 ... 92.5 92.75 93.0
    number      int64 8B 0
Data variables:
    ssrd        (valid_time, latitude, longitude) float32 5GB dask.array<chunksize=(720, 125, 121), meta=np.ndarray>
    strd        (valid_time, latitude, longitude) float32 5GB dask.array<chunksize=(720, 125, 121), meta=np.ndarray>
    t2m         (valid_time, latitude, longitude) float32 5GB dask.array<chunksize=(720, 125, 121), meta=np.ndarray>
    sst         (valid_time, latitude, longitude) float32 5GB dask.array<chunksize=(720, 125, 121), meta=np.ndarray>
    tcc         (valid_time, latitude, longitude) float32 5GB 

In [5]:
# Setting up Dask cluster

cluster = LocalCluster(
    n_workers=1,           
    memory_limit='10GB',   
    processes=False,      
    silence_logs='error'
)

client = Client(cluster)
print("Dask client connected:", client)

Dask client connected: <Client: 'inproc://192.168.29.14/13188/1' processes=1 threads=16, memory=9.31 GiB>


In [6]:
dir = os.path.join(os.getcwd(), 'flat_data_parquet')

var_ssrd = dd.read_parquet(os.path.join(dir, 'ssrd.parquet'))
var_strd = dd.read_parquet(os.path.join(dir, 'strd.parquet'))
var_tcc = dd.read_parquet(os.path.join(dir, 'tcc.parquet'))
var_lsm = dd.read_parquet(os.path.join(dir, 'lsm.parquet'))
var_z = dd.read_parquet(os.path.join(dir, 'z.parquet'))

print(f"SSRD partitions: {var_ssrd.npartitions}")
print(f"STRD partitions: {var_strd.npartitions}")
print(f"TCC partitions: {var_tcc.npartitions}")
print(f"LSM partitions: {var_lsm.npartitions}")
print(f"Z partitions: {var_z.npartitions}")

SSRD partitions: 12
STRD partitions: 26
TCC partitions: 14
LSM partitions: 10
Z partitions: 14


In [8]:
spatial_keys = ['latitude', 'longitude']
time_keys = ['valid_time', 'latitude', 'longitude']

ddf = var_ssrd.merge(var_strd, on=time_keys, how='left', shuffle_method='tasks')
ddf = ddf.merge(var_tcc, on=time_keys, how='left', shuffle_method='tasks')

first_time = var_lsm['valid_time'].head(1).iloc[0]
print(f"Using timestamp: {first_time}")

lsm_lookup = dd.read_parquet(
    'flat_data_parquet/lsm.parquet',
    filters=[('valid_time', '==', first_time)]
).compute()

z_lookup = dd.read_parquet(
    'flat_data_parquet/z.parquet',
    filters=[('valid_time', '==', first_time)]
).compute()

print(f'lsm lookup size: {len(lsm_lookup):}')
print(f'z lookup size: {len(z_lookup):}')

Using timestamp: 2016-01-01 00:00:00
lsm lookup size: 15125
z lookup size: 15125


In [ ]:
def merge_static(df_partition, lookup_df, keys):
    return df_partition.merge(lookup_df, on=keys, how='left')

ddf = ddf.map_partitions(merge_static, lsm_lookup, spatial_keys)
ddf = ddf.map_partitions(merge_static, z_lookup, spatial_keys)

ddf = ddf.rename(columns={
    'ssrd': 'solar_radiation',
    'strd': 'thermal_radiation',
    'tcc': 'cloud_cover',
    'lsm': 'land_sea_mask',
    'z': 'elevation'
})

for col in ['number', 'expver']:
    if col in ddf.columns:
        ddf = ddf.drop(col, axis=1)

print(f"Columns: {ddf.columns.tolist()}")
print(f"Partitions: {ddf.npartitions}")

ddf = ddf.persist()
print("Merge persisted across cluster.")

sample = ddf.head(100)
print(sample)